In [85]:
import pandas as pd

df = pd.read_csv('/kaggle/input/datasets/odins0n/top-20-play-store-app-reviews-daily-update/all_combined.csv')

df.head()

,reviewId,content,score,app
0,6275ac69-5290-4096-91d3-666c68af3026,good Facebook aap👍👍👍🤝🤝,5,Facebook
1,222ac664-66bd-42b4-8d88-ba95bdeb86f2,facebook is best app for touching with frnds a...,5,Facebook
2,1672ee05-508c-4ce3-87d6-e8b2db12b1f6,that's marvellous,5,Facebook
3,5f6565d9-4b8a-44fd-be2c-1f3d252ce43d,Community health and social mobilization worke...,5,Facebook
4,dd8f9ed7-1a02-408e-ab75-19f46339b774,nice social site,5,Facebook


**DATA UNDERSTANDING**

In [86]:
print("Total reviews:", len(df))
print("Total apps:", df['app'].nunique())
print("\nRatings distribution:")
print(df['score'].value_counts())

print("\nSample apps:")
print(df['app'].unique()[:10])

Total reviews: 200000
Total apps: 20

Ratings distribution:
score
5    122794
1     41789
4     17184
3     10045
2      8188
Name: count, dtype: int64

Sample apps:
['Facebook' 'WhatsApp' 'Facebook Messenger' 'Instagram' 'TikTok'
 'Subway Surfers' 'Facebook Lite' 'Microsoft Word' 'Microsoft PowerPoint'
 'Snapchat']


In [87]:
import re

def clean_text(text):
    text = str(text).lower()                      # lowercase
    text = re.sub(r'http\S+', '', text)           # remove URLs
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)    # remove special characters
    text = re.sub(r'\s+', ' ', text).strip()      # remove extra spaces
    return text

df = df.dropna(subset=["content", "score", "app"])

df["clean_review"] = df["content"].apply(clean_text)
df = df[df["clean_review"].str.len() > 20]  # remove too-short reviews
df[["content", "clean_review"]].head()

,content,clean_review
1,facebook is best app for touching with frnds a...,facebook is best app for touching with frnds a...
3,Community health and social mobilization worke...,community health and social mobilization worke...
5,Ma Muhammad Iqbal Af Pakistan sa Ho,ma muhammad iqbal af pakistan sa ho
8,facebook ake bohat achi app he mujhe ye bohat ...,facebook ake bohat achi app he mujhe ye bohat ...
10,this aap think its very nice but this is reall...,this aap think its very nice but this is reall...


In [88]:
grouped_df = df.groupby('app')['clean_review'].apply(list).reset_index()

grouped_df.head()

,app,clean_review
0,Candy Crush Saga,"[i this game its hard to put down, perfect way..."
1,Dropbox,[uploads stopped working just said i was offli...
2,Facebook,[facebook is best app for touching with frnds ...
3,Facebook Lite,[na restric account ko boiim paki ayus dinamna...
4,Facebook Messenger,[even though mine is updated others still cant...


In [89]:
import random

def create_training_samples(grouped_df, samples_per_app=200, bundle_size=10, seed=42):
    random.seed(seed)
    training_data = []

    for _, row in grouped_df.iterrows():
        app = row["app"]
        reviews = row["clean_review"]

        if len(reviews) < bundle_size:
            continue

        for _ in range(samples_per_app):
            selected = random.sample(reviews, bundle_size)
            training_data.append({"app": app, "reviews": " ".join(selected)})

    return pd.DataFrame(training_data)

training_df = create_training_samples(grouped_df, samples_per_app=200, bundle_size=10, seed=42)
training_df.head()

,app,reviews
0,Candy Crush Saga,it any only game i have on my phone but please...
1,Candy Crush Saga,super duper good game the game is fantabulous ...
2,Candy Crush Saga,the game is very interestingi so much love it ...
3,Candy Crush Saga,candy crush saga game is an interesting game i...
4,Candy Crush Saga,i love this game been playing for years great ...


**Build the AI Brain**

In [90]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_id = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

def flan_generate(prompt, max_new_tokens=64):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

print(flan_generate("Return JSON only: {\"sentiment\":\"positive\"}", max_new_tokens=40))

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


sentiment = "positive"


In [91]:
def flan_sentiment(reviews_text):
    reviews_text = str(reviews_text)[:600]
    prompt = f"""
Classify overall sentiment of these app reviews as one of:
positive, mixed, negative

Return only one word.

Reviews:
{reviews_text}
""".strip()
    out = flan_generate(prompt, max_new_tokens=5).strip().lower()
    if "pos" in out: return "positive"
    if "neg" in out: return "negative"
    return "mixed"

In [92]:
def flan_themes(reviews_text):
    reviews_text = str(reviews_text)[:600]
    prompt = f"""
Extract 5 key themes from these app reviews.
Return ONLY a comma-separated list of 5 short themes. No extra text.

Reviews:
{reviews_text}
""".strip()
    out = flan_generate(prompt, max_new_tokens=40).strip()
    themes = [t.strip() for t in out.split(",") if t.strip()]
    return themes[:5]

In [93]:
def flan_pain_points(reviews_text):
    reviews_text = str(reviews_text)[:600]
    prompt = f"""
List 5 pain points from these app reviews.
Return ONLY a comma-separated list. No extra text.

Reviews:
{reviews_text}
""".strip()
    out = flan_generate(prompt, max_new_tokens=60).strip()
    return [t.strip() for t in out.split(",") if t.strip()][:5]

In [94]:
def flan_delighters(reviews_text):
    reviews_text = str(reviews_text)[:600]
    prompt = f"""
List 5 things users like (delighters) from these app reviews.
Return ONLY a comma-separated list. No extra text.

Reviews:
{reviews_text}
""".strip()
    out = flan_generate(prompt, max_new_tokens=60).strip()
    return [t.strip() for t in out.split(",") if t.strip()][:5]

In [95]:
def flan_summary(reviews_text):
    reviews_text = str(reviews_text)[:600]
    prompt = f"""
Write ONE short sentence summarizing these app reviews.

Reviews:
{reviews_text}
""".strip()
    out = flan_generate(prompt, max_new_tokens=40).strip()
    return out

In [96]:
THEME_ACTION_MAP = [
    ("crash", ("Engineering","Fix crashes / stability","P0")),
    ("login", ("Engineering","Fix login/auth issues","P0")),
    ("update", ("Engineering","Investigate update regressions","P1")),
    ("ads", ("Product","Reduce ads frequency/placement","P1")),
    ("slow", ("Engineering","Improve performance/loading","P1")),
    ("lag", ("Engineering","Optimize to reduce lag","P1")),
    ("battery", ("Engineering","Investigate battery drain","P1")),
    ("payment", ("Engineering","Fix payment/subscription failures","P0")),
    ("refund", ("Support","Improve refund support process","P1")),
    ("subscription", ("Product","Improve subscription UX/value clarity","P2")),
    ("ui", ("Design","Improve UI clarity/navigation","P2")),
]

def suggest_actions_from_text(items):
    actions = []
    seen = set()
    for it in items:
        low = it.lower()
        for key, (owner, action, prio) in THEME_ACTION_MAP:
            if key in low:
                tup = (owner, action, prio)
                if tup not in seen:
                    seen.add(tup)
                    actions.append({"owner": owner, "action": action, "priority": prio})
    return actions[:5]

In [97]:
def analyze_reviews_engine(app, reviews_text):
    sent = flan_sentiment(reviews_text)
    themes = flan_themes(reviews_text)
    pains = flan_pain_points(reviews_text)
    dels  = flan_delighters(reviews_text)
    summ  = flan_summary(reviews_text)

    actions = suggest_actions_from_text(themes + pains)

    return {
        "sentiment": sent,
        "top_themes": [{"theme": t, "polarity": "mixed", "evidence": []} for t in themes],
        "pain_points": pains,
        "delighters": dels,
        "suggested_actions": actions,
        "summary": summ,
        "app": app
    }

In [98]:
sample = training_df.loc[0, "reviews"]
obj = analyze_reviews_engine("Candy Crush Saga", sample)
obj

{'sentiment': 'positive',
 'top_themes': [{'theme': 'i love this game',
   'polarity': 'mixed',
   'evidence': []}],
 'pain_points': ['consuming lot of data'],
 'delighters': ['i like this game all the mini games an competitions i always retur'],
 'suggested_actions': [],
 'summary': 'Gogogo is a great game for the whole family.',
 'app': 'Candy Crush Saga'}

In [99]:
import pandas as pd

batch = training_df.sample(30, random_state=42).reset_index(drop=True)

results = []
for i, row in batch.iterrows():
    app = row["app"]
    obj = analyze_reviews_engine(app, row["reviews"])
    obj["sample_id"] = i
    results.append(obj)

results_df = pd.DataFrame(results)
results_df[["app","sentiment","summary"]].head(10)

,app,sentiment,summary
0,Facebook,negative,... a little clunky and graphics are rough tho...
1,Twitter,positive,x is a powerful platform for writing and shari...
2,Facebook,positive,i enjoy to use this application it is very oka...
3,WhatsApp,positive,good but slow
4,Spotify,positive,i love this app
5,Candy Crush Saga,positive,Candy crush is a sweet and interesting game th...
6,Microsoft PowerPoint,positive,a good presentationmaking app
7,Dropbox,positive,a reliable and reliable app
8,Candy Crush Saga,positive,Candy crush saga is a fun and addicting game t...
9,Facebook Messenger,positive,When I first started working with the Almighty...


In [100]:
results_df.to_json("review_intel_outputs.jsonl", orient="records", lines=True, force_ascii=False)
results_df.to_csv("review_intel_outputs.csv", index=False)
print("✅ Saved outputs")

✅ Saved outputs


**Business Insights**

In [101]:
print(results_df["sentiment"].value_counts())

def explode_actions(df):
    rows = []
    for _, r in df.iterrows():
        for a in r.get("suggested_actions", []):
            rows.append({"app": r["app"], "owner": a["owner"], "priority": a["priority"]})
    return pd.DataFrame(rows)

actions_df = explode_actions(results_df)

print(actions_df["owner"].value_counts())
print(actions_df["priority"].value_counts())

sentiment
positive    18
negative    12
Name: count, dtype: int64
owner
Engineering    7
Design         1
Name: count, dtype: int64
priority
P1    5
P0    2
P2    1
Name: count, dtype: int64


This analysis is based on 30 sampled review bundles processed through the AI-powered Review Intelligence Engine.

The results indicate that overall user sentiment is largely positive, with 18 positive samples compared to 12 negative samples. This suggests that while users generally express satisfaction, a meaningful portion of feedback highlights areas that require improvement.

The majority of identified issues are technical in nature, with Engineering responsible for most recommended actions. This indicates that performance, stability, or functional challenges are the primary contributors to negative sentiment rather than design-related concerns.

From a prioritization standpoint, most issues fall under P1 (important), meaning they should be addressed in the near term. A smaller number of P0 (critical) issues require immediate attention, while very few are low-priority concerns.

Although this evaluation is based on a limited sample of 30 bundles, it demonstrates how AI-driven review intelligence can transform unstructured user feedback into structured, actionable insights — enabling teams to identify and respond to product issues significantly faster than traditional manual review processes.